In [0]:
dataset_path = "s3://dalhussein-courses/DE-Pro/datasets/bookstore/v1/"
sink_path = "dbfs:/Volumes/bookstore_ldp_catalog/landing/kafka_source"

def path_exists(path):
  try:
    dbutils.fs.ls(path)
    return True
  except Exception as e:
    msg = str(e)
    if ("com.databricks.sql.io.CloudFileNotFoundException" in msg
        or "java.io.FileNotFoundException" in msg):
      return False
    else:
      raise

def get_index(raw_dir):
    try:
        files = dbutils.fs.ls(raw_dir)
        file = max(f.name for f in files if f.name.endswith(".json"))
        index = int(file.split(".", maxsplit=1)[0])
        print(f"Current index is {index} and {file}")
    except:
        index = 0
    return index + 1
    
def load_json_file(current_idex, src_dir, raw_dir): 
    latest_file = f"{str(current_idex).zfill(2)}.json"
    print(latest_file)
    source = f"{src_dir}/{latest_file}"
    target = f"{raw_dir}/{latest_file}" 
    print(f"Streaming prefix is {src_dir}")
    prefix = src_dir.split("/")[-1]

    if path_exists(source):
        print(f"Loading {prefix}-{latest_file} file to the bookstore dataset")
        dbutils.fs.cp(source, target)
   
def __load_data(max, src_dir, raw_dir, all = False):
    index = get_index(raw_dir)
    if index > max:
        print("No more data to load \n")
        return 0
    elif all == True:
        while index <= max:
            load_json_file(index, src_dir, raw_dir)
            index += 1
    else:
        load_json_file(index, src_dir, raw_dir)
        index += 1
    return 1

def load_pipeline_data():     
    streaming_src = f"{dataset_path}/kafka-streaming/"
    raw_dir = f"{sink_path}/kafka-raw/"
    n = __load_data(10, streaming_src, raw_dir)

    books_streaming_dir = f"{dataset_path}/books-updates-streaming"
    books_raw_dir = f"{sink_path}/kafka-raw/books-updates"
    m = __load_data(5, books_streaming_dir, books_raw_dir)

    return n + m

In [0]:
num_files = load_pipeline_data()
dbutils.jobs.taskValues.set("num_new_files", num_files)